# Assignment 7: Forward-Backward Sweep Load Flow Analysis
## 1. Objective:
The objective of this assignment is to implement a Forward-Backward Sweep (FBS) load flow algorithm for a radial distribution system. The goal is to compute the bus voltages and branch currents of a given n-bus system.

---

## 2. Introduction
Power flow analysis is a fundamental task in power system operation and planning. Traditional load flow methods such as the Newton-Raphson or Gauss-Seidel algorithms are commonly used for transmission systems.
But for radial distribution systems, conventional methods like Newton-Raphson Method are less efficient due to:
1. High R/X ratio
2. Radial topology

Hence, the Forward-Backward Sweep method is preferred due to its simplicity and computational efficiency for radial systems.

---

## 3. System Description
The system considered is a radial distribution network consisting of multiple buses and branches.
Key Features:
1. Radial topology (no loops)
2. Single slack bus
3. PQ load buses
4. Line impedance data provided
5. Load data in MW and Mvar
6. Generation Data in MW and Mvar (not considered but implemented)
7. Reactive Power Compensation in Mvar (not considered but implemented)

---

## 4. Methodology
The Forward-Backward Sweep method consists of two main steps:

---

### 4.1. Backward Sweep
Starting from leaf nodes toward the slack bus, currents are computed.
$$ I_i = \frac{(P_i + jQ_i)^*}{V_i^*}$$
The total current at each node:
$$I_{node} = I_{load} + \sum I_{children}$$

---

### 4.2 Forward Sweep
Starting from the slack bus toward leaf nodes, voltages are updated:
$$V_{child} = V_{parent}-Z\cdot I_{child}$$

---

### 4.3 Convergence Criteria
The algorithm iterates until:
$$\max |V_{current} - V_{previous}| < \epsilon $$

---

## 5. Implementation Details
The algorithm was implemented in Python using an object-oriented approach.

**Key Components:**
- Bus Class
  Stores:
  - Voltage
  - Current
  - Load
  - Generator
  - Compensation
  - Branch Impedance (from Parent to Child)
  - Parent-Child Relations
- FBSSolver Class
  - Data extraction from Excel using Pandas
  - Network Construction
  - Traversal Generation
  - Backward and forward sweeps
  - Iterative Convergence

---

## 6. Network Construction
The radial distribution system is represented using a tree structure based on the given line data. 

Each row in the dataset specifies a connection between two buses in the form:
$$ \text{From Bus}\rightarrow\text{To Bus} $$

We assume in a radial system that the power flows from the source (slack bus) towards the downstream nodes. Therefore, we can consider:
- The **From Bus** is taken as the parent node.
- The **To Bus** is taken as the child node.

### 6.1 Algorithm
The following steps were used:
1. All unique bus numbers were identified from the **From** and **To** columns.
2. A Bus object was created for each node.
3. A tuple list was formed from the **From** and **To** columns and was iterated over.
4. For each tuple, ***i*** $\rightarrow$ ***j***,
   - Bus ***i*** was assigned as the **parent** of Bus ***j***
   - Bus ***j*** was added to the children list of Bus ***i***

### 6.2 Dictionary-Based Bus Objects
All bus objects were stored in a dictionary of the form:
$$\text{bus[i]} \rightarrow \text{Bus object corresponding to node i}$$
This allows direct access to any bus using its node number, regardless of what they are numbered.

---


## 7. Node Traversal Strategy
Instead of re-numbering nodes, traversal techniques were used to ensure proper computation order.

### 7.1 Backward Sweep Traversal
In this traversal, the current at each node depends on the currents of its child nodes. Hence, the child nodes must be processed first before their parent. This is achieved using **post-order traversal**, where:
- All child nodes are visited first.
- The parent node is processed afterward.

### 7.2 Forward Sweep Traversal
In this traversal, the voltage at a child node depends on the voltage of its parent node. Therefore, the parent must be processed first before its children.
This is achieved using pre-order traversal, where:
- The parent node is processed first
- Then its child nodes are visited

A list is generated for each traversal which is used during the iteration process.
This ensures correct computation without modifying the original node indices.

In [458]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [459]:
class Bus:
    def __init__(self,value):
        self.value= value
        self.parent = None
        self.children = []
        self.V = 1.0 + 0.0j
        self.I = 0.0 + 0.0j
        self.Pgen = 0.0 
        self.Qgen = 0.0
        self.Pload = 0.0 
        self.Qload = 0.0
        self.Qsh = 0.0
        self.Z = 0 + 1j * 0 # From PARENT -> CHILD

In [460]:
class FBSSolver:
    def __init__(self,fp,n,tol,Sb=1000,Vb=11):
        self.fp = fp
        self.n = n
        self.tol = tol
        self.Sb = Sb
        self.Vb = Vb
        self.fileValues(self.fp)
        self.__getNodes()
        self.__getTraversal()
        self.__solveFBS(self.n,self.tol)
        print("You can proceed with the Voltage and Current Data!!")

        
    def fileValues(self,fp):
        lineData = pd.read_excel(fp, sheet_name="Feeder Data")
        loadData = pd.read_excel(fp, sheet_name="Load Data")
        
        self.sBus = list(lineData['From'].astype(int))
        self.eBus = list(lineData['To'].astype(int))
        self.R = list(lineData['Res'].astype(float))
        self.X = list(lineData['Reac'].astype(float))
        
        self.Pload = list(loadData['MW'].astype(float))
        self.Qload = list(loadData['MVAR'].astype(float))
        self.bVal = list(loadData['From'].astype(int))
        print("Data Extracted Successfully")
    
    def __getNodes(self):
        tupArr = zip(self.sBus,self.eBus,self.R,self.X)
        all_nodes = np.unique(self.sBus + self.eBus)
        self.bus = {}
        for i in all_nodes:
            self.bus[int(i)] = Bus(int(i))
        for i,j,r,x in tupArr:
            self.bus[j].parent = self.bus[i]
            self.bus[i].children.append(self.bus[j])
            self.bus[j].Z = (r + 1j*x) #* ln
        for idx,i in enumerate(self.bVal):
            self.bus[i].Pload = self.Pload[idx]/(self.Sb)
            self.bus[i].Qload = self.Qload[idx]/(self.Sb)
        print("System Formed")
    
    def __getTraversal(self):
        self.back_sweep = []
        def postOrder(node):
            for child in node.children:
                postOrder(child)
            self.back_sweep.append(node.value)
        self.fwd_sweep = []
        def preOrder(node):
            self.fwd_sweep.append(node.value)
            for child in node.children:
                preOrder(child)
        postOrder(self.bus[1])
        preOrder(self.bus[1])
        print("Traversals Obtained")

    def __backwardSweep(self):
        for i in self.back_sweep:
            cbus = self.bus[i]
            Snet = (cbus.Pload-cbus.Pgen) + 1j * (cbus.Qload - cbus.Qgen + cbus.Qsh)
            I_load = np.conjugate(Snet) / np.conjugate(cbus.V)
            cbus.I = I_load
            for child in cbus.children:
                cbus.I = cbus.I + child.I
    
    def __forwardSweep(self):
        for i in self.fwd_sweep:
            pbus = self.bus[i]
            for child in pbus.children:
                child.V = pbus.V - child.I * child.Z

    def __getV(self,x):
        yV = np.zeros(len(x)+1,dtype=complex)
        for i in x:
            yV[i] = x[i].V
        return yV
    
    def __solveFBS(self,n,t):
        flag = 0
        for itr in range(n):
            Vprev = self.__getV(self.bus)
            self.__backwardSweep()
            self.__forwardSweep()
            Vcurr = self.__getV(self.bus)
            if max(abs(Vcurr-Vprev))<t:
                print("Convergence Successful @ Iteration {}".format(itr))
                flag = 1
                break
        if flag == 0:
            print("Solver Compeleted")
    
    def getValues(self):
        Vbus = np.array([self.bus[i].V for i in self.bus])
        Ibus = np.array([self.bus[i].I for i in self.bus])
        Z = np.array([self.bus[i].Z for i in self.bus], dtype=complex)
        return Vbus,Ibus,Z

    def displayNodes(self):
        for i in self.bus:
            b = self.bus[i]
            if b.parent == None: 
                z = 'None' 
            else: 
                z = b.parent.value 
            stmt = """ 
            Bus {a} 
            Parent: {b} 
            Children: {c} 
            Branch Impedance: {d} """.format(a = b.value, b = z, c = [j.value for j in b.children], d = b.Z) 
            print(stmt)
            
    def displayInfo(self):
        V,I,Z = self.getValues()
        print()
        print("Output Results at each Node of the Radial Network:\n")
        print('Bus  Vmag     Angle  Pgen_kW  Qgen_kVAR   Pload_kW  Qload_kVAR  Qsh_kVAR\n')
        for i in self.bus:
            b = self.bus[i]
            print(f"{i:>2d}  {abs(b.V):>6.3f}  {np.angle(b.V,deg=True):>7.3f} \
 {(b.Pgen) * self.Sb:>7f}  {b.Qgen * self.Sb:>9f}  {b.Pload * self.Sb:>9.2f}  {b.Qload * self.Sb:>9.2f} {b.Qsh * self.Sb:>10f}")

    def displayLosses(self):
        for i in self.bus:
            b = self.bus[i]
            stmt = """
            Node {}: 
                Active Loss -> {}
                Reactive Loss -> {}

    

In [461]:
itr = 50
tol = 1e-9
Sb = 1000 #kVA
Vb = 11 #kV
sys1 = FBSSolver("19-Node-Radial-Network.xlsx",itr,tol,Sb,Vb)
V,I,Z = sys1.getValues()
Ploss = np.sum(np.square(abs(I)) * np.real(Z)) #* 1000
Qloss = np.sum(np.square(abs(I)) * np.imag(Z)) #* 1000
dev = max(abs(1-V))
sys1.displayInfo()
print()
print("Maximum Deviation (pu): {}".format(dev))
print("Active Power Loss (MW): {}".format(Ploss))
print("Reactive Power Loss (Mvar): {}".format(Qloss))

Data Extracted Successfully
System Formed
Traversals Obtained
Convergence Successful @ Iteration 43
You can proceed with the Voltage and Current Data!!

Output Results at each Node of the Radial Network:

Bus  Vmag     Angle  Pgen_kW  Qgen_kVAR   Pload_kW  Qload_kVAR  Qsh_kVAR

 1   1.000    0.000  0.000000   0.000000       0.00       0.00   0.000000
 2   0.906    0.176  0.000000   0.000000     108.00      52.31   0.000000
 3   0.897    0.200  0.000000   0.000000     144.00      69.74   0.000000
 4   0.863    0.266  0.000000   0.000000      36.00      17.44   0.000000
 5   0.861    0.271  0.000000   0.000000      90.00      43.59   0.000000
 6   0.836    0.326  0.000000   0.000000      36.00      17.44   0.000000
 7   0.832    0.337  0.000000   0.000000     144.00      69.74   0.000000
 8   0.774    0.474  0.000000   0.000000     108.00      52.31   0.000000
 9   0.704    0.669  0.000000   0.000000     216.00     104.61   0.000000
10   0.603    1.019  0.000000   0.000000      54.00    

## 8. Observations
### 8.1 Voltage Profile
1. The slack bus voltage remains fixed at 1.0 pu
2. Voltages decrease progressively along the feeder
3. The minimum voltage occurs at buses farthest from the slack bus, which is bus 19

### 8.2 Maximum Voltage Deviation
1. Voltage deviation increases with distance from the slack
2. Maximum deviation observed: 0.4458 pu

### 8.3 Total Losses in the System
1. Branches with larger currents will have higher losses
2. Total Active Power: 1.047 MW
3. Total Reactive Power: 0.450 Mvar

In [462]:
sys1.displayNodes()

 
            Bus 1 
            Parent: None 
            Children: [2] 
            Branch Impedance: 0j 
 
            Bus 2 
            Parent: 1 
            Children: [3, 4] 
            Branch Impedance: (0.0258+0.0111j) 
 
            Bus 3 
            Parent: 2 
            Children: [] 
            Branch Impedance: (0.043+0.0185j) 
 
            Bus 4 
            Parent: 2 
            Children: [5, 6] 
            Branch Impedance: (0.0129+0.00555j) 
 
            Bus 5 
            Parent: 4 
            Children: [] 
            Branch Impedance: (0.0129+0.00555j) 
 
            Bus 6 
            Parent: 4 
            Children: [7, 8] 
            Branch Impedance: (0.0086+0.0037j) 
 
            Bus 7 
            Parent: 6 
            Children: [] 
            Branch Impedance: (0.0172+0.0074j) 
 
            Bus 8 
            Parent: 6 
            Children: [9] 
            Branch Impedance: (0.0215+0.00925j) 
 
            Bus 9 
            Parent: 8 
       